In [ ]:
!pip install nltk scikit-learn seaborn scipy

import pandas as pd
import numpy as np
import nltk
import re
import seaborn as sns
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

from scipy.sparse import hstack

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/CFPB_Consumer_Complaints_2024.csv')
# Use only required columns
df = df[['consumer_complaint_narrative', 'product']]
df = df.dropna()

# Rename columns
df.columns = ['text', 'label']

print(df.shape)
df.head()


(200, 2)


,text,label
3,"During my time as an employee for XXXX, I was ...",Debt collection
9,I have been monitoring my credit weekly for 2 ...,Debt collection
11,I previously had phone service with XXXX XXXX....,Debt collection
13,Several debt collectors have called me on beha...,Debt collection
16,On XX/XX/2020 I settled and paid a debt with E...,Debt collection


Basic preprocessing

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  # remove numbers/punctuation
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text'] = df['text'].apply(clean_text)

Handle class imbalance (IMPORTANT)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])

In [ ]:
# Remove classes with very few samples
counts = df['label'].value_counts()

# Keep only classes with >= 2 samples (better: >= 5)
valid_labels = counts[counts >= 5].index

df = df[df['label'].isin(valid_labels)]

Train-test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['text'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

Apply SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_res, y_train_res = smote.fit_resample(X_train_tfidf, y_train)

model training

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=200)
lr_model.fit(X_train_res, y_train_res)

LogisticRegression(max_iter=200)

In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train_res, y_train_res)

MultinomialNB()

In [ ]:
from sklearn.svm import SVC

svm_model = SVC(kernel='linear', probability=True)
svm_model.fit(X_train_res, y_train_res)

SVC(kernel='linear', probability=True)

In [ ]:
from sklearn.ensemble import VotingClassifier

ensemble = VotingClassifier(
    estimators=[
        ('lr', lr_model),
        ('nb', nb_model),
        ('svm', svm_model)
    ],
    voting='soft'
)

ensemble.fit(X_train_res, y_train_res)

VotingClassifier(estimators=[('lr', LogisticRegression(max_iter=200)),
                             ('nb', MultinomialNB()),
                             ('svm', SVC(kernel='linear', probability=True))],
                 voting='soft')

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

def evaluate(model, name):
    y_pred = model.predict(X_test_tfidf)
    print(f"\n{name} Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))

evaluate(lr_model, "Logistic Regression")
evaluate(nb_model, "Naive Bayes")
evaluate(svm_model, "SVM")
evaluate(ensemble, "Ensemble")


Logistic Regression Accuracy: 0.75
              precision    recall  f1-score   support

           1       0.00      0.00      0.00         7
           2       0.81      0.91      0.86        33

    accuracy                           0.75        40
   macro avg       0.41      0.45      0.43        40
weighted avg       0.67      0.75      0.71        40


Naive Bayes Accuracy: 0.7
              precision    recall  f1-score   support

           1       0.14      0.14      0.14         7
           2       0.82      0.82      0.82        33

    accuracy                           0.70        40
   macro avg       0.48      0.48      0.48        40
weighted avg       0.70      0.70      0.70        40


SVM Accuracy: 0.8
              precision    recall  f1-score   support

           1       0.00      0.00      0.00         7
           2       0.82      0.97      0.89        33

    accuracy                           0.80        40
   macro avg       0.41      0.48      0.44   

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
def predict_complaint(text):
    text = text.lower()
    vec = vectorizer.transform([text])
    pred = ensemble.predict(vec)

    return le.inverse_transform(pred)[0]

print(predict_complaint("My loan payment is not updated"))

Debt collection


In [ ]:
import pickle

# Save trained models
pickle.dump(ensemble, open("model.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))
pickle.dump(le, open("label_encoder.pkl", "wb"))

print("✅ All models saved successfully!")

✅ All models saved successfully!


In [ ]:
model = pickle.load(open("model.pkl", "rb"))
vectorizer = pickle.load(open("vectorizer.pkl", "rb"))
le = pickle.load(open("label_encoder.pkl", "rb"))

In [ ]:
from google.colab import files

files.download("model.pkl")
files.download("vectorizer.pkl")
files.download("label_encoder.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>